In [2]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import spacy
from spacy.matcher import DependencyMatcher
from spacy.displacy import parse_deps
import spacy.displacy as displacy
import sys
import tqdm
import random

sys.path.append("../src")

from shared import tqdm_readlines

In [3]:
deu_nlp = spacy.load("de_zdl_lg")

In [4]:
eng_snts = dict([(ln.strip().split("\t")[0], ln.strip().split("\t")[1]) for ln in tqdm_readlines("../data/tatoeba/eng.tsv")])

links = [ln.strip().split(",") for ln in tqdm_readlines("../data/tatoeba/links.csv")]

100%|███████████████████████████████████████████████████| 2689446/2689446 [00:01<00:00, 1460602.09it/s]


In [5]:
deu_snts = dict([(ln.strip().split("\t")[0], deu_nlp(ln.strip().split("\t")[1])) for (ln, _) in zip(tqdm_readlines("../data/tatoeba/deu.tsv"), range(10000))])
eng_deu_pairs = [(eng_snts[l[0]], deu_snts[l[1]]) for r in links for l in [(r[0], r[1]), (r[1], r[0])] if l[0] in eng_snts if l[1] in deu_snts]

  3%|█▍                                                       | 10000/384458 [04:38<2:53:59, 35.87it/s]


In [16]:
#s = eng_rus_pairs[5003][1]
#len(eng_rus_pairs)

# print(sorted(list(set([t.dep_ for sp in eng_rus_pairs for t in sp[1]]))))

# s = deu_nlp("Ich freue mich auf die Reise.")
# s = deu_nlp("Dafür interessiere ich mich nicht.")
# s = deu_nlp("Dafür habe ich mich nicht interessiert.")
# s = deu_nlp("Es riecht nach Käse.")
# s = deu_nlp("Dabei wurde ein 34-jähriger Mann schwer verletzt.")
s = deu_nlp("Ich bin an stark gewürztes Essen nicht gewöhnt.")
print(parse_deps(s))

{'words': [{'text': 'Ich', 'tag': 'PRON', 'lemma': None}, {'text': 'bin', 'tag': 'AUX', 'lemma': None}, {'text': 'an', 'tag': 'ADP', 'lemma': None}, {'text': 'stark', 'tag': 'ADJ', 'lemma': None}, {'text': 'gewürztes', 'tag': 'ADJ', 'lemma': None}, {'text': 'Essen', 'tag': 'NOUN', 'lemma': None}, {'text': 'nicht', 'tag': 'PART', 'lemma': None}, {'text': 'gewöhnt.', 'tag': 'VERB', 'lemma': None}], 'arcs': [{'start': 0, 'end': 7, 'label': 'nsubj:pass', 'dir': 'left'}, {'start': 1, 'end': 7, 'label': 'aux:pass', 'dir': 'left'}, {'start': 2, 'end': 5, 'label': 'case', 'dir': 'left'}, {'start': 3, 'end': 4, 'label': 'advmod', 'dir': 'left'}, {'start': 4, 'end': 5, 'label': 'amod', 'dir': 'left'}, {'start': 5, 'end': 7, 'label': 'obl', 'dir': 'left'}, {'start': 6, 'end': 7, 'label': 'advmod', 'dir': 'left'}], 'settings': {'lang': 'de', 'direction': 'ltr'}}


In [7]:
def search_nlp(nlp, ptn):
    matcher = DependencyMatcher(nlp.vocab)
    matcher.add("PTN", [ptn])
    return matcher

mtch = search_nlp(deu_nlp, [{
    "RIGHT_ID": "anchor", 
    "RIGHT_ATTRS": { "POS": "VERB" }
}, {
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "obl_obj", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": "obl"
    }
}, {
    "LEFT_ID": "obl_obj", 
    "REL_OP": ">", 
    "RIGHT_ID": "obl_prep", 
    "RIGHT_ATTRS": {
        "POS": "ADP",
        "DEP": "case"
    }
}])

mtch(s)

[]

In [8]:
combos = Counter()

for s_eng, s_deu in eng_deu_pairs:
    m = mtch(s_deu)
    if len(m) > 0:
        # print(i, s, m, s[m[0][1][0]])
        # print(m)
        # print(m)
        # print(s_deu.text)
        vb = s_deu[m[0][1][0]].lemma_
        adp = s_deu[m[0][1][2]].lemma_
        # print(s_deu[m[0][1][0]].lemma_, "+", s_deu[m[0][1][2]], "+", s_deu[m[0][1][1]], " | ", s_deu.text)
        combos[(vb, adp)] += 1
        # print(i, s[m[0][1][1]], "+", s[m[0][1][0]], "|", s)

for (k, c) in combos.most_common():
    print(k, c)

('gehen', 'in') 118
('gehen', 'zu') 86
('kommen', 'zu') 56
('essen', 'zu') 50
('geben', 'in') 46
('fahren', 'mit') 46
('sehen', 'in') 46
('bitten', 'um') 42
('helfen', 'bei') 34
('liegen', 'auf') 34
('kommen', 'nach') 34
('kommen', 'in') 32
('interessieren', 'für') 32
('tun', 'mit') 30
('liegen', 'in') 28
('stehen', 'in') 28
('passen', 'zu') 24
('arbeiten', 'in') 24
('finden', 'in') 22
('erinnern', 'an') 22
('gehen', 'nach') 22
('stehen', 'auf') 20
('bleiben', 'zu') 20
('studieren', 'in') 20
('leben', 'in') 20
('bleiben', 'in') 18
('machen', 'in') 18
('machen', 'an') 18
('schwimmen', 'in') 18
('bringen', 'zu') 18
('wohnen', 'in') 18
('gehen', 'an') 16
('treffen', 'auf') 16
('brechen', 'in') 16
('treffen', 'in') 16
('gehen', 'über') 16
('stehen', 'um') 16
('verletzen', 'bei') 16
('kaufen', 'in') 16
('gehen', 'vor') 16
('geben', 'zu') 14
('halten', 'an') 14
('stehen', 'vor') 14
('gehen', 'auf') 14
('sterben', 'vor') 14
('halten', 'von') 14
('bringen', 'in') 14
('überfahren', 'von') 14
('